# 04 — Neural Feature Extraction

Compute log-mel spectrograms and their deltas to produce 1-channel and 3-channel neural inputs from preprocessed 1 s segments (16 kHz, mono). Shapes: (99, 40, 1) and (99, 40, 3).

In [1]:
# Imports and setup
from pathlib import Path
import yaml, json
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
CFG_PATH = PROJECT_ROOT / 'config.yaml'
UNIV_DIR = PROJECT_ROOT / 'data' / 'processed' / 'universal'
OUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'neural_features'
OUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR = PROJECT_ROOT / 'results' / 'metrics'
METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)


Project root: /Users/harryirving/Development/projects/ai-ml/BikeAIv4


In [2]:
# Load config (supports  or config:) and set parameters
with open(CFG_PATH, 'r') as f:
    cfg = yaml.safe_load(f)
data_cfg = (cfg.get('data') or cfg.get('config', {}).get('data') or cfg.get('config') or {})
audio_cfg = (cfg.get('audio') or cfg.get('config', {}).get('audio') or {})
TARGET_SR = int(audio_cfg.get('sample_rate', 16000))
# Mel-spectrogram parameters following training plan
N_MELS = 40
N_FFT = 512
HOP = 160  # ~10 ms hop at 16 kHz to target ~99 frames per 1 s
FMIN = 80  # align with 80 Hz HPF
FMAX = TARGET_SR // 2
TARGET_T = 99
CLASSES = ['angle_grinder', 'background', 'tools']
AUDIO_EXTS = ('.wav',)
print('Universal dir:', UNIV_DIR)
assert UNIV_DIR.exists(), f'Missing directory: {UNIV_DIR}'


Universal dir: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/data/processed/universal


## Utilities

In [3]:
def to_mono(y):
    if y.ndim == 1:
        return y
    return librosa.to_mono(y.T)

def ensure_sr(y, sr, target_sr=16000):
    if sr == target_sr:
        return y, sr
    y2 = librosa.resample(y, orig_sr=sr, target_sr=target_sr, res_type='soxr_hq')
    return y2, target_sr

def fix_time_length(M, target_T=99):
    # M shape: (n_mels, T)
    T = M.shape[1]
    if T == target_T:
        return M
    if T < target_T:
        pad_total = target_T - T
        left = pad_total // 2
        right = pad_total - left
        return np.pad(M, ((0,0),(left,right)), mode='constant', constant_values=np.min(M))
    # center-crop
    start = (T - target_T) // 2
    return M[:, start:start+target_T]

def per_sample_standardize(X, eps=1e-6):
    mu = np.mean(X, axis=(0,1), keepdims=True)
    sd = np.std(X, axis=(0,1), keepdims=True)
    return (X - mu) / (sd + eps)

def compute_logmel(y, sr, n_fft=N_FFT, hop=HOP, n_mels=N_MELS, fmin=FMIN, fmax=FMAX):
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, hop_length=hop, n_mels=n_mels, fmin=fmin, fmax=fmax, power=2.0, center=True)
    S_db = librosa.power_to_db(S, ref=np.max)
    return S_db


## Extraction loop

In [4]:
X1_list, X3_list, y_list, paths = [], [], [], []
for label in CLASSES:
    cls_dir = UNIV_DIR / label
    if not cls_dir.exists():
        print('Skipping missing class dir:', cls_dir)
        continue
    files = sorted([p for p in cls_dir.rglob('*') if p.suffix.lower() in AUDIO_EXTS])
    print(f'Extracting {label}: {len(files)} files')
    for p in tqdm(files):
        try:
            y, sr = sf.read(p)
            if y.ndim > 1:
                y = to_mono(y)
            y = y.astype(np.float32, copy=False)
            y, sr = ensure_sr(y, sr, TARGET_SR)
            M_db = compute_logmel(y, sr)  # (n_mels, T) in dB
            M_db = fix_time_length(M_db, TARGET_T)
            # Transpose to (T, n_mels) then standardize per-sample
            M_T = M_db.T.astype(np.float32)
            M_T = per_sample_standardize(M_T)
            # Deltas computed on mel (dB) before transpose for stability
            Delta = librosa.feature.delta(M_db, order=1)
            Delta2 = librosa.feature.delta(M_db, order=2)
            Delta = fix_time_length(Delta, TARGET_T).T.astype(np.float32)
            Delta2 = fix_time_length(Delta2, TARGET_T).T.astype(np.float32)
            # 1-channel and 3-channel stacks
            x1 = M_T[..., None]                 # (T, 40, 1)
            x3 = np.stack([M_T, Delta, Delta2], axis=-1)  # (T, 40, 3)
            X1_list.append(x1)
            X3_list.append(x3)
            y_list.append(label)
            paths.append(str(p))
        except Exception as e:
            print('Error:', p, e)
            continue
X1 = np.array(X1_list, dtype=np.float32) if X1_list else np.empty((0, TARGET_T, N_MELS, 1), dtype=np.float32)
X3 = np.array(X3_list, dtype=np.float32) if X3_list else np.empty((0, TARGET_T, N_MELS, 3), dtype=np.float32)
y = np.array(y_list)
print('Shapes — X1:', X1.shape, 'X3:', X3.shape, 'y:', y.shape)


Extracting angle_grinder: 6378 files


  0%|          | 0/6378 [00:00<?, ?it/s]

Extracting background: 8069 files


  0%|          | 0/8069 [00:00<?, ?it/s]

Extracting tools: 4597 files


  0%|          | 0/4597 [00:00<?, ?it/s]

Shapes — X1: (19044, 99, 40, 1) X3: (19044, 99, 40, 3) y: (19044,)


## Save arrays and manifest

In [5]:
out_1ch = OUT_DIR / 'spectrograms_1ch.npy'
out_3ch = OUT_DIR / 'spectrograms_3ch.npy'
out_y = OUT_DIR / 'labels.npy'
np.save(out_1ch, X1)
np.save(out_3ch, X3)
np.save(out_y, y)
manifest = pd.DataFrame({'path': paths, 'label': y_list})
manifest.to_csv(METRICS_DIR / 'neural_features_manifest.csv', index=False)
print('Saved:', out_1ch)
print('Saved:', out_3ch)
print('Saved labels:', out_y)
print('Saved manifest:', METRICS_DIR / 'neural_features_manifest.csv')


Saved: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/data/processed/neural_features/spectrograms_1ch.npy
Saved: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/data/processed/neural_features/spectrograms_3ch.npy
Saved labels: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/data/processed/neural_features/labels.npy
Saved manifest: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/results/metrics/neural_features_manifest.csv


## Quick spot-check

In [6]:
# Visualize a random example from each class (if available)
import matplotlib.pyplot as plt
import random
def show_example(label_target):
    idxs = [i for i, lbl in enumerate(y_list) if lbl == label_target]
    if not idxs:
        print('No examples for', label_target)
        return
    i = random.choice(idxs)
    fig, ax = plt.subplots(1,3, figsize=(12,3))
    ax[0].imshow(X1[i][...,0].T, origin='lower', aspect='auto', cmap='magma')
    ax[0].set_title(f'{label_target} — log-mel')
    ax[1].imshow(X3[i][...,1].T, origin='lower', aspect='auto', cmap='magma')
    ax[1].set_title('Delta')
    ax[2].imshow(X3[i][...,2].T, origin='lower', aspect='auto', cmap='magma')
    ax[2].set_title('Delta-Delta')
    plt.tight_layout()
    out = FIG_DIR / f'spotcheck_neural_{label_target}.png'
    plt.savefig(out, dpi=150); plt.close(fig)
    print('Saved figure:', out)
for cls in CLASSES:
    show_example(cls)


Saved figure: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/results/figures/spotcheck_neural_angle_grinder.png
Saved figure: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/results/figures/spotcheck_neural_background.png
Saved figure: /Users/harryirving/Development/projects/ai-ml/BikeAIv4/results/figures/spotcheck_neural_tools.png
